# Support Vector Machines (SVM) Algorithm

This notebook provides a university-level introduction to the Support Vector Machine algorithm, covering both theoretical foundations and practical implementations in Python.

Author: Michał Woźniak

## Technical Setup: Running This Notebook Locally with `uv`

### Prerequisites
- Ensure you have `uv` installed on your system. If not, install it following the instructions at [https://github.com/astral-sh/uv](https://github.com/astral-sh/uv)

### Step 1: Initialize a New Project
Navigate to your working directory and initialize a new Python project:
```bash
# Navigate to your project directory
cd /path/to/your/project

# Initialize uv project (if not already initialized)
uv init

# Or if you want to specify Python version
uv init --python 3.11
```

### Step 2: Install Required Dependencies
Install all necessary packages for this notebook:
```bash
# Install core data science packages
uv add numpy pandas scikit-learn scipy

# Install visualization libraries
uv add matplotlib seaborn

# Install Jupyter notebook
uv add jupyter ipykernel

# Alternative: Install all at once
uv add numpy pandas scikit-learn scipy matplotlib seaborn jupyter ipykernel
```

### Step 3: Activate the Environment and Start Jupyter
```bash
# Activate the virtual environment (uv handles this automatically)
# Start Jupyter notebook
uv run jupyter notebook

# Or use Jupyter Lab
uv run jupyter lab
```

### Step 4: Open This Notebook
Once Jupyter starts, navigate to `SVM.ipynb` and open it.

### Required Packages Summary
- **numpy** (≥1.21.0): Numerical computations and array operations
- **pandas** (≥1.3.0): Data manipulation and analysis
- **scikit-learn** (≥1.0.0): Machine learning algorithms and utilities
- **scipy** (≥1.7.0): Scientific computing (quadratic programming solver for the dual SVM)
- **matplotlib** (≥3.4.0): Basic plotting and visualization
- **seaborn** (≥0.11.0): Statistical data visualization
- **jupyter**: Jupyter notebook interface
- **ipykernel**: Jupyter kernel for Python

### Troubleshooting
- If you encounter import errors, ensure all packages are installed: `uv pip list`
- To update packages: `uv add --upgrade package_name`
- To see your Python version: `uv run python --version`


# 1. Introduction to Support Vector Machines

## What is a Support Vector Machine?

A **Support Vector Machine (SVM)** is a **supervised**, **max-margin**, **kernel-based** learning algorithm used for both classification and regression tasks.

### Key Characteristics:

1. **Max-margin classifier**: Instead of finding *any* hyperplane that separates the classes, SVM looks for the hyperplane whose distance to the closest training points (the *support vectors*) is as large as possible. This margin tends to give better generalization.

2. **Parametric but geometrically motivated**: After training, the SVM stores only a set of weights (or a set of support vectors in the kernel case) — not the entire training set like KNN.

3. **Kernel trick**: SVMs can learn highly non-linear decision boundaries by implicitly mapping the data into a higher-dimensional feature space through a *kernel function*, without ever computing the mapping explicitly.

### How SVM Works:

The core idea is geometric:
- **For Classification**: Find the hyperplane that separates the two classes with the largest possible margin. Only the points closest to the boundary — the *support vectors* — determine the solution.
- **For Regression (SVR)**: Fit a function that deviates from the true targets by at most $\varepsilon$, while being as flat as possible.

### Mathematical Foundation

#### Linear SVM — Primal Formulation

For a binary classification problem with labels $y_i \in \{-1,+1\}$, a linear classifier uses the decision function

$$f(x) = w^\top x + b$$

and predicts the class by the sign of $f(x)$:

$$\hat{y} = \operatorname{sign}(f(x))$$

The decision boundary is therefore the hyperplane

$$w^\top x + b = 0$$

where $w$ is the normal vector to the hyperplane and $b$ is the bias term.

---

#### Hard-Margin SVM

If the data is perfectly linearly separable, SVM chooses the separating hyperplane with the **largest margin**. The hard-margin optimization problem is

$$\min_{w,b}\; \frac{1}{2}\|w\|^2 \qquad \text{s.t.} \qquad y_i(w^\top x_i + b) \ge 1 \quad \forall i$$

The constraint

$$y_i(w^\top x_i + b) \ge 1$$

means:

* if $y_i = +1$, then $w^\top x_i + b \ge 1$
* if $y_i = -1$, then $w^\top x_i + b \le -1$

So the training points must lie outside the two margin boundaries

$$w^\top x + b = 1 \qquad \text{and} \qquad w^\top x + b = -1$$

with the separating hyperplane $w^\top x + b = 0$ exactly halfway between them.

---

#### Why the Margin Width Is $\frac{2}{\|w\|}$

The two margin boundaries are parallel hyperplanes with common normal vector $w$.
For two parallel hyperplanes of the form

$$w^\top x = c_1 \qquad \text{and} \qquad w^\top x = c_2$$

the perpendicular distance between them is

$$\frac{|c_1 - c_2|}{\|w\|}$$

In our case,

$$w^\top x + b = 1 \;\Longrightarrow\; w^\top x = 1 - b$$

$$w^\top x + b = -1 \;\Longrightarrow\; w^\top x = -1 - b$$

so the difference in constants is $2$. Therefore the margin width is

$$\frac{2}{\|w\|}$$

and the distance from the decision boundary $w^\top x + b = 0$ to either margin boundary is

$$\frac{1}{\|w\|}$$

Hence minimizing $\|w\|^2$ is equivalent to maximizing the margin.

---

#### Geometric Interpretation

SVM does not merely search for any separating hyperplane. It searches for the one that leaves the **widest possible safety buffer** between the two classes. A larger margin generally improves robustness and generalization, because small perturbations of the input are less likely to change the predicted class.

The points that lie exactly on the margin, or violate it in the soft-margin setting, are called **support vectors**. These are the critical training examples that determine the position of the classifier.

---

#### Soft-Margin SVM (Real Data)

Real data is rarely perfectly separable. To allow margin violations, we introduce slack variables $\xi_i \ge 0$ and a regularization constant $C > 0$:

$$\min_{w,b,\xi}\; \frac{1}{2}\|w\|^2 + C\sum_{i=1}^n \xi_i \qquad \text{s.t.} \qquad y_i(w^\top x_i + b) \ge 1 - \xi_i,\;\; \xi_i \ge 0$$

The slack variable $\xi_i$ measures how much point $x_i$ violates the ideal margin condition:

* $\xi_i = 0$: correctly classified and on or outside the margin
* $0 < \xi_i < 1$: correctly classified but inside the margin
* $\xi_i = 1$: lies exactly on the decision boundary
* $\xi_i > 1$: misclassified

Thus the soft-margin SVM balances two competing goals:

1. keeping the margin large by minimizing $\|w\|^2$
2. keeping violations small by penalizing $\sum_i \xi_i$

---

#### Hinge Loss Formulation

The soft-margin problem can be written equivalently as unconstrained minimization with the **hinge loss**:

$$\min_{w,b}\; \frac{1}{2}\|w\|^2 + C\sum_{i=1}^n \max\bigl(0,\;1 - y_i(w^\top x_i + b)\bigr)$$

For each point, define the signed score

$$m_i = y_i(w^\top x_i + b)$$

Then the hinge loss is

$$\max(0, 1 - m_i)$$

which behaves as follows:

* if $m_i \ge 1$, the loss is $0$
* if $m_i < 1$, the loss is $1 - m_i$

So there is no penalty for points correctly classified with sufficient margin, while points inside the margin or misclassified are penalized linearly.

---

#### Role of the Regularization Parameter $C$

The parameter $C$ controls the tradeoff between margin size and training violations:

* **Small $C$**: violations are penalized weakly  
  $\rightarrow$ wider margin, more violations allowed, stronger regularization

* **Large $C$**: violations are penalized strongly  
  $\rightarrow$ narrower margin, fewer violations allowed, weaker regularization

Equivalently:

* small $C$: prioritize a simpler, more robust decision boundary
* large $C$: prioritize fitting the training data more closely

A very large $C$ can make the model sensitive to noise and outliers, while a very small $C$ can lead to underfitting.

---

#### Dual Formulation and the Kernel Trick

Using Lagrangian duality, the soft-margin SVM can be rewritten as

$$\max_{\alpha}\; \sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i=1}^n\sum_{j=1}^n \alpha_i \alpha_j y_i y_j \langle x_i, x_j \rangle$$

subject to

$$0 \le \alpha_i \le C, \qquad \sum_{i=1}^n \alpha_i y_i = 0$$

This dual form is important because the data appears only through inner products

$$\langle x_i, x_j \rangle$$

rather than directly through coordinates.

The primal weight vector can be recovered as

$$w = \sum_{i=1}^n \alpha_i y_i x_i$$

so only training points with $\alpha_i > 0$ contribute to the classifier. These are precisely the **support vectors**.

---

#### Support Vectors and the Meaning of $\alpha_i$

The dual variables $\alpha_i$ indicate which training points matter:

* $\alpha_i = 0$: point does not affect the solution
* $0 < \alpha_i < C$: point lies exactly on the margin
* $\alpha_i = C$: point is inside the margin or misclassified

This explains why SVMs are often sparse: only a subset of training points determines the decision boundary.

The prediction function can therefore be written as

$$f(x) = \sum_{i \in SV} \alpha_i y_i \langle x_i, x \rangle + b$$

where $SV$ denotes the set of support vectors.

---

#### The Kernel Trick

Since the dual depends only on inner products, we can replace

$$\langle x_i, x_j \rangle$$

with a kernel function

$$K(x_i, x_j)$$

which computes the inner product in some feature space $\phi(x)$:

$$K(x, x') = \langle \phi(x), \phi(x') \rangle$$

This allows SVMs to learn non-linear decision boundaries without explicitly mapping data into a high-dimensional feature space.

The resulting classifier becomes

$$f(x) = \operatorname{sign}\!\left(\sum_{i \in SV} \alpha_i y_i K(x_i, x) + b\right)$$

This is the **kernel trick**: perform linear separation in an implicit feature space, which corresponds to a nonlinear boundary in the original input space.

---

#### Common Kernels

1. **Linear kernel**

$$K(x, x') = x^\top x'$$

This gives the standard linear SVM. It is efficient and works well when the data is approximately linearly separable or when the number of features is very large.

2. **Polynomial kernel** (degree $d$)

$$K(x, x') = (\gamma\, x^\top x' + r)^d$$

This allows curved decision boundaries and interactions up to degree $d$.

3. **Radial Basis Function (RBF / Gaussian) kernel**

$$K(x, x') = \exp\!\bigl(-\gamma \|x - x'\|^2\bigr)$$

This is the most widely used nonlinear kernel. It can represent highly flexible decision boundaries by measuring similarity based on distance.

4. **Sigmoid kernel**

$$K(x, x') = \tanh(\gamma\, x^\top x' + r)$$

This is related to neural-network-style activations, though it is used less often in practice.

---

#### Kernel Parameters and Model Complexity

For nonlinear kernels, hyperparameters also control complexity:

* In the **RBF kernel**, $\gamma$ determines how localized the similarity is:
  * small $\gamma$: smoother, less complex boundary
  * large $\gamma$: more flexible, highly local boundary

* In the **polynomial kernel**, the degree $d$ controls the complexity of feature interactions.

Thus, in kernel SVMs, model complexity is influenced by both:

* $C$, which controls tolerance to violations
* kernel hyperparameters such as $\gamma$ or $d$


## Key Hyperparameters

### 1. Regularization Parameter (C)
- **Small C**: Large margin, tolerates more misclassifications → more regularization, higher bias
- **Large C**: Small margin, tries to classify every training point correctly → less regularization, higher variance
- **Best practice**: Use cross-validation to find the right balance

### 2. Kernel
- Choice depends on the nature of your data and the shape of the expected decision boundary
- **Linear**: Works well when data is (approximately) linearly separable, and is much faster on high-dimensional sparse data (e.g. text)
- **RBF**: Good default for most non-linear problems
- **Polynomial**: Useful when features interact through known polynomial relationships

### 3. Gamma (for RBF / polynomial / sigmoid kernels)
- Controls how far the influence of a single training example reaches
- **Low gamma**: Far-reaching influence → smoother, more biased boundary
- **High gamma**: Short-range influence → boundary closely follows individual training points → risk of overfitting
- A common starting point is `gamma='scale'` (1 / (n_features * X.var())) in scikit-learn

### 4. Degree (for polynomial kernel)
- The degree of the polynomial
- Higher degrees can model more complex boundaries but are more prone to overfitting and numerical issues

## Decision Boundaries

SVMs create **maximum-margin** decision boundaries:

- With a **linear kernel**, the boundary is a hyperplane — geometrically the widest possible slab between the classes.
- With an **RBF kernel**, the boundary can be arbitrarily curved — each support vector contributes a localized "bump" to the decision function.
- The parameters $C$ and $\gamma$ together control the bias-variance trade-off:
  - Small $C$, small $\gamma$ → smooth, underfitting
  - Large $C$, large $\gamma$ → wiggly, overfitting


## Advantages and Limitations

### Advantages ✓
1. **Effective in high-dimensional spaces**: Works well even when the number of features is greater than the number of samples
2. **Memory efficient at prediction time**: Only support vectors are needed for the final decision function
3. **Versatile through kernels**: Can model linear and highly non-linear relationships with the same algorithm
4. **Strong theoretical guarantees**: Based on statistical learning theory (VC dimension, structural risk minimization)
5. **Unique global optimum**: The underlying convex optimization problem has no local minima

### Limitations ✗
1. **Training is expensive on large datasets**: Kernel SVM is roughly O(n²) to O(n³) in training time, making it impractical for very large n
2. **Does not produce probabilistic outputs natively**: Probabilities require an extra calibration step (e.g. Platt scaling)
3. **Sensitive to the choice of C and gamma**: Requires careful hyperparameter tuning
4. **Requires feature scaling**: Distance-based kernels (like RBF) are dominated by features with large variance
5. **Multi-class not native**: SVM is inherently binary; multi-class is handled via One-vs-Rest or One-vs-One wrappers
6. **Hard to interpret**: With non-linear kernels, the model is essentially a black box

## The Role of Support Vectors

Only a subset of training points ends up with non-zero dual coefficients — these are the **support vectors**. They lie either *on* the margin or *inside* it (for the soft-margin case). All other points could be removed from the training set without changing the learned model.

This sparsity is what makes SVMs elegant: the decision function depends only on a (usually small) subset of the data.

## Critical Preprocessing: Feature Scaling

**Why is feature scaling essential for SVM?**

Consider this example:
- Feature 1: Age (range 20-80)
- Feature 2: Income (range 20,000-200,000)

Without scaling, the RBF kernel $\exp(-\gamma \|x - x'\|^2)$ is dominated by the income difference — the age feature effectively disappears. Even a linear kernel becomes numerically ill-conditioned when feature scales differ by orders of magnitude.

**Common scaling methods:**

1. **Min-Max Scaling (Normalization)**: Scales to [0, 1]
   $$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

2. **Standardization (Z-score normalization)**: Zero mean, unit variance
   $$x' = \frac{x - \mu}{\sigma}$$

3. **Robust Scaling**: Uses median and IQR (robust to outliers)
   $$x' = \frac{x - \text{median}}{IQR}$$

**Best practice**: Apply the same scaling to both training and test data using parameters learned from training data only!


# 2. From-Scratch Implementation: Classification

In this section, we'll implement SVM from scratch to understand the underlying mechanics. We'll build:
1. A **linear SVM** trained in the **primal** via sub-gradient descent on the hinge loss.
2. A **kernel SVM** trained in the **dual** via a quadratic-programming solver from SciPy.

Then we'll compare their behavior on the Iris dataset.

## 2.1 Linear SVM (Primal, Sub-gradient Descent)

Recall that we want to minimize:
$$\mathcal{L}(w, b) = \frac{1}{2}\|w\|^2 + C \sum_{i=1}^{n} \max\bigl(0, \; 1 - y_i(w^\top x_i + b)\bigr)$$

The sub-gradient of the hinge term with respect to $w$ is:
$$\frac{\partial}{\partial w}\,\max(0, 1 - y_i(w^\top x_i + b)) = \begin{cases} -y_i x_i & \text{if } y_i(w^\top x_i + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$

The sub-gradient with respect to $b$ is:
$$\frac{\partial}{\partial b}\,\max(0, 1 - y_i(w^\top x_i + b)) = \begin{cases} -y_i & \text{if } y_i(w^\top x_i + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$

We update $w$ and $b$ using this sub-gradient.


In [ ]:
# Import necessary libraries
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib configuration for better plots
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
# Load the Iris dataset
from sklearn.datasets import load_iris

# Load data
iris = load_iris()
X = iris.data
y = iris.target

print("Iris Dataset Loaded")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature names: {iris.feature_names}")
print(f"Target names: {iris.target_names}")
print(f"\nFirst 5 samples:\n{X[:5]}")
print(f"First 5 labels: {y[:5]}")

# SVM is inherently a BINARY classifier. For the from-scratch sections we
# will use a two-class subset: versicolor (1) vs virginica (2). This pair
# is NOT perfectly linearly separable, so it nicely showcases the role of
# the soft-margin and the kernel trick.
binary_mask = y != 0  # drop setosa
X_bin = X[binary_mask]
y_bin = np.where(y[binary_mask] == 1, -1, 1)  # versicolor=-1, virginica=+1

print(f"\nBinary subset shape: {X_bin.shape}")
print(f"Label distribution: {np.unique(y_bin, return_counts=True)}")

### Step 1: Implement the Hinge Loss and its Sub-gradient

In [ ]:
def hinge_loss(w, b, X, y, C):
    """
    Compute the full soft-margin SVM objective:
        0.5 * ||w||^2 + C * sum_i max(0, 1 - y_i (w.x_i + b))

    Parameters
    ----------
    w : array-like, shape (n_features,)
    b : float
    X : array-like, shape (n_samples, n_features)
    y : array-like, shape (n_samples,), values in {-1, +1}
    C : float
        Regularization strength (larger C -> less regularization).

    Returns
    -------
    float : objective value
    """
    margins = 1 - y * (X @ w + b)  # np.
    hinge = np.maximum(0, margins)
    return 0.5 * np.dot(w, w) + C * np.sum(hinge)


def hinge_subgradient(w, b, X, y, C):
    """
    Compute the sub-gradient of the hinge-loss SVM objective
    with respect to w and b.

    Returns
    -------
    grad_w : array, shape (n_features,)
    grad_b : float
    """
    # Which samples violate the margin? (1 - y*(w.x+b) > 0)
    margins = 1 - y * (X @ w + b)
    active = margins > 0  # boolean mask of violating samples

    grad_w = w - C * (y[active] @ X[active])
    grad_b = -C * np.sum(y[active])
    return grad_w, grad_b


# Quick sanity check
w_test = np.array([0.5, -0.2])
b_test = 0.1
X_test_hinge = np.array([[1.0, 2.0], [-1.0, -1.0]])
y_test_hinge = np.array([1, -1])
print(f"Loss:  {hinge_loss(w_test, b_test, X_test_hinge, y_test_hinge, C=1.0):.4f}")
gw, gb = hinge_subgradient(w_test, b_test, X_test_hinge, y_test_hinge, C=1.0)
print(f"grad_w: {gw}")
print(f"grad_b: {gb}")

### Step 2: Implement a Linear SVM Classifier from Scratch

In [ ]:
class LinearSVMClassifier:
    """
    Linear Support Vector Machine trained in the primal via sub-gradient
    descent on the soft-margin hinge-loss objective.

    Assumes binary labels in {-1, +1}.
    """

    def __init__(self, C=1.0, learning_rate=0.001, n_iters=2000):
        """
        Parameters
        ----------
        C : float
            Regularization strength. Larger C -> less regularization
            (fits training data harder, smaller margin).
        learning_rate : float
            Step size for gradient descent.
        n_iters : int
            Number of gradient-descent iterations.
        """
        self.C = C
        self.learning_rate = learning_rate
        self.n_iters = n_iters

    def fit(self, X_train, y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float)
        n_samples, n_features = X_train.shape

        # Initialize parameters
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.loss_history_ = []

        for it in range(self.n_iters):
            grad_w, grad_b = hinge_subgradient(self.w, self.b, X_train, y_train, self.C)
            self.w -= self.learning_rate * grad_w
            self.b -= self.learning_rate * grad_b

            if it % 100 == 0:
                self.loss_history_.append(hinge_loss(self.w, self.b, X_train, y_train, self.C))
        return self

    def decision_function(self, X):
        return np.asarray(X) @ self.w + self.b

    def predict(self, X):
        return np.sign(self.decision_function(X))

    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)


print("LinearSVMClassifier class created successfully!")

### Step 3: Split Data and Apply Feature Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the binary data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42, stratify=y_bin)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Feature scaling is CRITICAL for SVM!
scaler = StandardScaler()  # z-score = (x - mean) / std
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nBefore scaling (first 3 training samples):")
print(X_train[:3])
print("\nAfter scaling (first 3 training samples):")
print(X_train_scaled[:3])

In [ ]:
scaler.mean_, scaler.scale_

### Step 4: Train and Test the Linear SVM Model

In [ ]:
# Create and train the model
svm = LinearSVMClassifier(C=1.0, learning_rate=0.001, n_iters=2000)

start_time = time.time()
svm.fit(X_train_scaled, y_train)
train_time = time.time() - start_time

# Measure prediction time
start_time = time.time()
y_pred = svm.predict(X_test_scaled)
prediction_time = time.time() - start_time

# Calculate accuracy
accuracy = svm.score(X_test_scaled, y_test)

print(f"Regularization (C):  {svm.C}")
print(f"Learning rate:       {svm.learning_rate}")
print(f"Iterations:          {svm.n_iters}")
print(f"Training time:       {train_time:.4f} seconds")
print(f"Prediction time:     {prediction_time:.4f} seconds")
print(f"Accuracy:            {accuracy:.4f} ({accuracy * 100:.2f}%)")

print(f"\nLearned w: {svm.w}")
print(f"Learned b: {svm.b:.4f}")
print("\nSample predictions (first 10):")
print(f"Predicted: {y_pred[:10].astype(int)}")
print(f"Actual:    {y_test[:10]}")

# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(svm.loss_history_[1:])) * 100, svm.loss_history_[1:], "o-")
plt.xlabel("Iteration")
plt.ylabel("Objective value")
plt.title("Linear SVM - Hinge Loss during Training")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Step 5: Evaluate with Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["versicolor", "virginica"],
)
disp.plot(cmap="Blues", ax=ax)
plt.title("Confusion Matrix - Linear SVM (From Scratch)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Confusion Matrix:")
print(cm)

### Step 6: Visualize Decision Boundaries (2D)

In [ ]:
# Use only first 2 features for visualization
X_bin_2d = X_bin[:, :2]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_bin_2d, y_bin, test_size=0.3, random_state=42, stratify=y_bin
)

# Scale the 2D features
scaler_2d = StandardScaler()
X_train_2d_scaled = scaler_2d.fit_transform(X_train_2d)
X_test_2d_scaled = scaler_2d.transform(X_test_2d)

# Train Linear SVM on 2D data
svm_2d = LinearSVMClassifier(C=1.0, learning_rate=0.001, n_iters=3000)
svm_2d.fit(X_train_2d_scaled, y_train_2d)

# Create mesh for decision boundary
h = 0.02  # step size in the mesh
x_min, x_max = X_train_2d_scaled[:, 0].min() - 1, X_train_2d_scaled[:, 0].max() + 1
y_min, y_max = X_train_2d_scaled[:, 1].min() - 1, X_train_2d_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predict decision function over the mesh
Z_dec = svm_2d.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z_dec = Z_dec.reshape(xx.shape)
Z = np.sign(Z_dec)

# Plot decision boundaries + margin lines
fig, ax = plt.subplots(figsize=(12, 8))
ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm", levels=[-2, 0, 2])
# Decision boundary and margins: f(x) = 0, f(x) = +/-1
ax.contour(xx, yy, Z_dec, levels=[-1, 0, 1], colors=["k", "k", "k"], linestyles=["--", "-", "--"], linewidths=[1, 2, 1])

scatter = ax.scatter(
    X_train_2d_scaled[:, 0],
    X_train_2d_scaled[:, 1],
    c=y_train_2d,
    cmap="coolwarm",
    edgecolor="black",
    s=100,
    alpha=0.8,
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Class", fontsize=12)
cbar.set_ticks([-1, 1])
cbar.set_ticklabels(["versicolor", "virginica"])

ax.set_xlabel(iris.feature_names[0], fontsize=12)
ax.set_ylabel(iris.feature_names[1], fontsize=12)
ax.set_title("Linear SVM Decision Boundary and Margins (C=1.0) - From Scratch", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Calculate accuracy on 2D data
accuracy_2d = svm_2d.score(X_test_2d_scaled, y_test_2d)
print(f"Accuracy with only 2 features: {accuracy_2d:.4f} ({accuracy_2d * 100:.2f}%)")

## 2.2 Kernel SVM (Dual Formulation)

### Why go to the dual?

In the primal formulation we optimize over $w$ directly — but $w$ lives in the same space as the input features. If we want to use a non-linear feature map $\phi(x)$ that sends data into a very high (or infinite!) dimensional space, we cannot represent $w$ explicitly.

The dual formulation avoids this issue. It only ever touches the data through **inner products** $\langle x_i, x_j\rangle$, which we can replace with a kernel $K(x_i, x_j)$ — the famous **kernel trick**.

#### Dual optimization problem

$$\max_{\alpha} \; \sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i=1}^{n}\sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j K(x_i, x_j)$$
$$\text{subject to} \quad 0 \le \alpha_i \le C, \quad \sum_{i=1}^{n} \alpha_i y_i = 0$$

#### Prediction

$$f(x) = \text{sign}\!\left(\sum_{i=1}^{n} \alpha_i y_i K(x_i, x) + b\right)$$

Only the $\alpha_i > 0$ matter — those training points are the **support vectors**.

#### Complexity
- **Training**: dominated by solving the QP, roughly $O(n^2)$ to $O(n^3)$ depending on the solver
- **Prediction**: $O(|SV| \cdot d)$ per query — depends on the number of support vectors
- **Memory**: stores the support vectors and their coefficients

#### When to use Kernel SVMs:
- Small to medium datasets (up to ~10k-50k samples)
- Non-linear decision boundaries expected
- When a simple feature map is not obvious but similarity is easy to define


### Kernel SVM Implementation using SciPy's QP Solver

In [ ]:
from scipy.optimize import minimize


class KernelSVMClassifier:
    """
    Kernel Support Vector Machine trained via the dual formulation.

    The dual QP is solved with scipy.optimize.minimize (SLSQP). This
    implementation is pedagogical — it scales to a few hundred samples,
    not to production-size datasets. Use sklearn.svm.SVC for real work.
    """

    def __init__(self, C=1.0, kernel="rbf", gamma=0.5, degree=3, coef0=1.0):
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0

    # ---- kernel functions ----
    def _kernel(self, X1, X2):
        if self.kernel == "linear":
            return X1 @ X2.T
        if self.kernel == "rbf":
            sq = np.sum(X1**2, axis=1)[:, None] + np.sum(X2**2, axis=1)[None, :] - 2 * X1 @ X2.T
            return np.exp(-self.gamma * sq)
        if self.kernel == "poly":
            return (self.gamma * (X1 @ X2.T) + self.coef0) ** self.degree
        raise ValueError(f"Unknown kernel: {self.kernel}")

    def fit(self, X_train, y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float)
        n = len(X_train)

        # Pre-compute kernel (Gram) matrix
        K = self._kernel(X_train, X_train)
        # Outer product of labels -- appears in the dual objective
        P = np.outer(y_train, y_train) * K

        # Dual objective: maximize sum(alpha) - 0.5 alpha^T P alpha
        # --> minimize 0.5 alpha^T P alpha - sum(alpha)
        def objective(alpha):
            return 0.5 * alpha @ P @ alpha - np.sum(alpha)

        def jacobian(alpha):
            return P @ alpha - np.ones(n)

        # Constraints: sum(alpha_i * y_i) = 0,  0 <= alpha_i <= C
        constraints = {
            "type": "eq",
            "fun": lambda a: np.dot(a, y_train),
            "jac": lambda a: y_train,
        }
        bounds = [(0.0, self.C)] * n
        alpha0 = np.full(n, self.C / 2.0)

        result = minimize(
            objective,
            alpha0,
            jac=jacobian,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 200, "ftol": 1e-8},
        )
        self.alpha_ = result.x

        # Identify support vectors (alpha strictly positive)
        sv_mask = self.alpha_ > 1e-5
        self.sv_X_ = X_train[sv_mask]
        self.sv_y_ = y_train[sv_mask]
        self.sv_alpha_ = self.alpha_[sv_mask]

        # Compute bias b using margin support vectors (0 < alpha < C)
        margin_mask = (self.alpha_ > 1e-5) & (self.alpha_ < self.C - 1e-5)
        if margin_mask.any():
            # For any margin SV: y_k = sum_i alpha_i y_i K(x_i, x_k) + b
            f_margin = (self.alpha_ * y_train) @ K[:, margin_mask]
            self.b_ = np.mean(y_train[margin_mask] - f_margin)
        else:
            self.b_ = 0.0

        self.n_support_ = int(sv_mask.sum())
        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=float)
        K = self._kernel(X, self.sv_X_)
        return K @ (self.sv_alpha_ * self.sv_y_) + self.b_

    def predict(self, X):
        return np.sign(self.decision_function(X))

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


print("KernelSVMClassifier class created successfully!")

### Performance Comparison: Linear (Primal) vs Kernel (Dual)

In [ ]:
# Train Kernel SVM (RBF)
svm_kernel = KernelSVMClassifier(C=1.0, kernel="rbf", gamma=0.5)

start_time = time.time()
svm_kernel.fit(X_train_scaled, y_train)
kernel_train_time = time.time() - start_time

start_time = time.time()
y_pred_kernel = svm_kernel.predict(X_test_scaled)
kernel_pred_time = time.time() - start_time

accuracy_kernel = svm_kernel.score(X_test_scaled, y_test)

# Re-measure Linear SVM times for a fair comparison
start_time = time.time()
svm_linear = LinearSVMClassifier(C=1.0, learning_rate=0.001, n_iters=2000)
svm_linear.fit(X_train_scaled, y_train)
linear_train_time = time.time() - start_time

start_time = time.time()
y_pred_linear = svm_linear.predict(X_test_scaled)
linear_pred_time = time.time() - start_time

accuracy_linear = svm_linear.score(X_test_scaled, y_test)

print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"\n{'Method':<25} {'Train Time (s)':<15} {'Predict Time (s)':<18} {'Accuracy':<10}")
print("-" * 60)
print(f"{'Linear SVM (primal)':<25} {linear_train_time:<15.6f} {linear_pred_time:<18.6f} {accuracy_linear:<10.4f}")
print(f"{'Kernel SVM (RBF dual)':<25} {kernel_train_time:<15.6f} {kernel_pred_time:<18.6f} {accuracy_kernel:<10.4f}")
print("-" * 60)

print(f"\nNumber of support vectors (Kernel SVM): {svm_kernel.n_support_} / {len(X_train_scaled)}")
print("\nNote: On linearly-separable-ish data a linear SVM may match an RBF one,")
print("      while for highly non-linear data the kernel version will dominate.")

### Visualizing the Kernel SVM Decision Boundary (2D)

In [ ]:
# Train Kernel SVM on 2D data
svm_kernel_2d = KernelSVMClassifier(C=1.0, kernel="rbf", gamma=0.8)
svm_kernel_2d.fit(X_train_2d_scaled, y_train_2d)

# Decision boundary visualization
h = 0.02
x_min, x_max = X_train_2d_scaled[:, 0].min() - 1, X_train_2d_scaled[:, 0].max() + 1
y_min, y_max = X_train_2d_scaled[:, 1].min() - 1, X_train_2d_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z_dec = svm_kernel_2d.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z_dec = Z_dec.reshape(xx.shape)
Z = np.sign(Z_dec)

fig, ax = plt.subplots(figsize=(12, 8))
ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm", levels=[-2, 0, 2])
ax.contour(xx, yy, Z_dec, levels=[-1, 0, 1], colors=["k", "k", "k"], linestyles=["--", "-", "--"], linewidths=[1, 2, 1])

scatter = ax.scatter(
    X_train_2d_scaled[:, 0],
    X_train_2d_scaled[:, 1],
    c=y_train_2d,
    cmap="coolwarm",
    edgecolor="black",
    s=100,
    alpha=0.8,
)

# Highlight support vectors
ax.scatter(
    svm_kernel_2d.sv_X_[:, 0],
    svm_kernel_2d.sv_X_[:, 1],
    s=200,
    facecolors="none",
    edgecolors="yellow",
    linewidths=2,
    label="Support vectors",
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Class", fontsize=12)
cbar.set_ticks([-1, 1])
cbar.set_ticklabels(["versicolor", "virginica"])

ax.set_xlabel(iris.feature_names[0], fontsize=12)
ax.set_ylabel(iris.feature_names[1], fontsize=12)
ax.set_title("Kernel SVM Decision Boundary (RBF, C=1, gamma=0.8) - From Scratch", fontsize=14, fontweight="bold")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

accuracy_kernel_2d = svm_kernel_2d.score(X_test_2d_scaled, y_test_2d)
print(f"Kernel SVM Accuracy with 2 features: {accuracy_kernel_2d:.4f} ({accuracy_kernel_2d * 100:.2f}%)")
print(f"Number of support vectors: {svm_kernel_2d.n_support_} / {len(X_train_2d_scaled)}")

# 3. Scikit-learn Implementation

Now that we understand how SVMs work internally, let's use scikit-learn's optimized implementation (built on top of the highly efficient `libsvm` library) for both classification and regression tasks.

## 3.1 SVM Classification with Scikit-learn

We'll use the full (3-class) Iris dataset and explore comprehensive evaluation metrics. `SVC` handles multi-class classification automatically using the One-vs-One strategy.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.svm import SVC

# Use the full, 3-class Iris dataset from here on
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

# Create and train the model
svm_sklearn = SVC(C=1.0, kernel="rbf", gamma="scale")
svm_sklearn.fit(X_train_full_scaled, y_train_full)

# Make predictions
y_pred_sklearn = svm_sklearn.predict(X_test_full_scaled)

# Calculate metrics
accuracy = accuracy_score(y_test_full, y_pred_sklearn)
precision = precision_score(y_test_full, y_pred_sklearn, average="weighted")
recall = recall_score(y_test_full, y_pred_sklearn, average="weighted")
f1 = f1_score(y_test_full, y_pred_sklearn, average="weighted")

print("=" * 60)
print("SCIKIT-LEARN SVM CLASSIFICATION RESULTS")
print("=" * 60)
print(f"\nAccuracy:  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"\nNumber of support vectors per class: {svm_sklearn.n_support_}")

print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test_full, y_pred_sklearn, target_names=iris.target_names))

In [ ]:
# Visualize confusion matrix
cm_sklearn = confusion_matrix(y_test_full, y_pred_sklearn)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_sklearn, display_labels=iris.target_names)
disp.plot(cmap="Blues", ax=ax, values_format="d")
plt.title("Confusion Matrix - Scikit-learn SVM", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Print confusion matrix interpretation
print("\nConfusion Matrix Interpretation:")
print(f"Total correct predictions: {np.trace(cm_sklearn)}")
print(f"Total incorrect predictions: {cm_sklearn.sum() - np.trace(cm_sklearn)}")

### Comparing Different C Values

In [ ]:
# Test different C values on a logarithmic grid (that is how C should be searched)
C_values = np.logspace(-3, 3, 13)  # 0.001 .. 1000
train_scores = []
test_scores = []

for C_val in C_values:
    svc = SVC(C=C_val, kernel="rbf", gamma="scale")
    svc.fit(X_train_full_scaled, y_train_full)
    train_scores.append(svc.score(X_train_full_scaled, y_train_full))
    test_scores.append(svc.score(X_test_full_scaled, y_test_full))

# Plot the results (log x-axis because C spans orders of magnitude)
fig, ax = plt.subplots(figsize=(12, 6))
ax.semilogx(C_values, train_scores, "o-", label="Training Accuracy", linewidth=2, markersize=6)
ax.semilogx(C_values, test_scores, "s-", label="Test Accuracy", linewidth=2, markersize=6)
ax.set_xlabel("Regularization parameter C (log scale)", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("SVM Classification: Accuracy vs. C", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find optimal C
optimal_C = C_values[int(np.argmax(test_scores))]
best_test_accuracy = max(test_scores)
print(f"Optimal C: {optimal_C:.4f}")
print(f"Best test accuracy: {best_test_accuracy:.4f} ({best_test_accuracy * 100:.2f}%)")
print("\nNote: As C increases, the model fits the training data harder")
print("      (higher variance, lower bias). Small C -> wider margin, more regularization.")

## 3.2 SVM Regression with Scikit-learn (SVR)

SVMs can also be used for regression tasks via **Support Vector Regression (SVR)**. Instead of classifying, SVR tries to fit a function whose deviation from each training target is at most $\varepsilon$, while keeping the model as flat (small $\|w\|$) as possible. Points outside the $\varepsilon$-tube become support vectors; points inside do not contribute.

We'll use the California Housing dataset to demonstrate SVR.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.svm import SVR

# Load California Housing dataset
housing = fetch_california_housing()
X_housing = housing.data
y_housing = housing.target

print("California Housing Dataset Loaded")
print(f"Features shape: {X_housing.shape}")
print(f"Target shape: {y_housing.shape}")
print(f"\nFeature names: {housing.feature_names}")
print("Target: Median house value (in $100,000s)")
print(f"\nFirst 5 samples (features):\n{X_housing[:5]}")
print(f"First 5 targets: {y_housing[:5]}")

In [ ]:
# Use a subset for faster computation -- kernel SVR is O(n^2 .. n^3)
# In practice, you would use something like a linear SVR or a completely
# different model on the full dataset.
from sklearn.utils import resample

X_housing_sample, y_housing_sample = resample(X_housing, y_housing, n_samples=5000, random_state=42)

# Split the data
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_housing_sample, y_housing_sample, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_h.shape[0]}")
print(f"Test set size: {X_test_h.shape[0]}")

# Feature scaling is ESSENTIAL for SVR!
scaler_h = StandardScaler()
X_train_h_scaled = scaler_h.fit_transform(X_train_h)
X_test_h_scaled = scaler_h.transform(X_test_h)

print("\nData split and scaled successfully!")

In [ ]:
# Train SVR
svr = SVR(C=1.0, kernel="rbf", gamma="scale", epsilon=0.1)
svr.fit(X_train_h_scaled, y_train_h)

# Make predictions
y_pred_h = svr.predict(X_test_h_scaled)

# Calculate regression metrics
mse = mean_squared_error(y_test_h, y_pred_h)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_h, y_pred_h)
r2 = r2_score(y_test_h, y_pred_h)

print("=" * 60)
print("SVR REGRESSION RESULTS")
print("=" * 60)
print(f"\nMean Squared Error (MSE):       {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE):      {mae:.4f}")
print(f"R^2 Score:                       {r2:.4f}")
print(f"\nNumber of support vectors:      {svr.support_.shape[0]} / {len(X_train_h_scaled)}")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print(f"On average, predictions are off by ${mae * 100000:.2f}")
print(f"The model explains {r2 * 100:.2f}% of the variance in house prices")

In [ ]:
# Visualize predictions vs actual values
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot: Predicted vs Actual
axes[0].scatter(y_test_h, y_pred_h, alpha=0.5, s=30)
axes[0].plot(
    [y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], "r--", lw=2, label="Perfect Prediction"
)
axes[0].set_xlabel("Actual Values ($100k)", fontsize=12)
axes[0].set_ylabel("Predicted Values ($100k)", fontsize=12)
axes[0].set_title("SVR: Predicted vs Actual", fontsize=14, fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = y_test_h - y_pred_h
axes[1].scatter(y_pred_h, residuals, alpha=0.5, s=30)
axes[1].axhline(y=0, color="r", linestyle="--", lw=2)
axes[1].set_xlabel("Predicted Values ($100k)", fontsize=12)
axes[1].set_ylabel("Residuals", fontsize=12)
axes[1].set_title("Residual Plot", fontsize=14, fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Good residual plots show:")
print("1. Random scatter around zero (no pattern)")
print("2. Constant variance across all predicted values")
print("3. Most points close to the zero line")

In [ ]:
# Compare different C values for SVR
C_values_reg = np.logspace(-2, 2, 9)  # 0.01 .. 100
train_r2_scores = []
test_r2_scores = []
train_rmse_scores = []
test_rmse_scores = []

for C_val in C_values_reg:
    svr_k = SVR(C=C_val, kernel="rbf", gamma="scale", epsilon=0.1)
    svr_k.fit(X_train_h_scaled, y_train_h)

    # R^2
    train_r2_scores.append(svr_k.score(X_train_h_scaled, y_train_h))
    test_r2_scores.append(svr_k.score(X_test_h_scaled, y_test_h))

    # RMSE
    y_pred_train_k = svr_k.predict(X_train_h_scaled)
    train_rmse_scores.append(np.sqrt(mean_squared_error(y_train_h, y_pred_train_k)))

    y_pred_k = svr_k.predict(X_test_h_scaled)
    test_rmse_scores.append(np.sqrt(mean_squared_error(y_test_h, y_pred_k)))

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# R^2 scores
axes[0].semilogx(C_values_reg, train_r2_scores, "o-", label="Training R^2", linewidth=2, markersize=6)
axes[0].semilogx(C_values_reg, test_r2_scores, "s-", label="Test R^2", linewidth=2, markersize=6)
axes[0].set_xlabel("Regularization parameter C (log scale)", fontsize=12)
axes[0].set_ylabel("R^2 Score", fontsize=12)
axes[0].set_title("SVR: R^2 Score vs. C", fontsize=14, fontweight="bold")
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# RMSE
axes[1].semilogx(C_values_reg, train_rmse_scores, "o-", label="Training RMSE", linewidth=2, markersize=6)
axes[1].semilogx(C_values_reg, test_rmse_scores, "s-", label="Test RMSE", color="red", linewidth=2, markersize=6)
axes[1].set_xlabel("Regularization parameter C (log scale)", fontsize=12)
axes[1].set_ylabel("RMSE", fontsize=12)
axes[1].set_title("SVR: RMSE vs. C", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find optimal C
optimal_C_reg = C_values_reg[int(np.argmax(test_r2_scores))]
best_r2 = max(test_r2_scores)
best_rmse = min(test_rmse_scores)
print(f"Optimal C for SVR: {optimal_C_reg:.4f}")
print(f"Best test R^2 score: {best_r2:.4f}")
print(f"Best test RMSE: {best_rmse:.4f}")

# 4. Cross-Validation for Model Evaluation

Cross-validation is a crucial technique for assessing how well a model generalizes to unseen data. It helps us avoid overfitting and provides more reliable performance estimates than a single train-test split.

## Why Cross-Validation?

**Problem with single train-test split:**
- Results can vary significantly depending on which samples end up in training vs. test
- May get lucky (or unlucky) with the split
- Doesn't fully utilize the available data for training

**Solution: Cross-Validation**
- Systematically rotate through different train-test combinations
- More robust performance estimate
- Better use of limited data
- Helps detect overfitting/underfitting

## 4.1 Hold-out Approach (Baseline)

The simplest approach: split data once into train and test sets.


In [ ]:
# We've already done this earlier, but let's formalize it
print("=" * 60)
print("HOLD-OUT APPROACH")
print("=" * 60)

# Split data
X_train_ho, X_test_ho, y_train_ho, y_test_ho = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale features
scaler_ho = StandardScaler()
X_train_ho_scaled = scaler_ho.fit_transform(X_train_ho)
X_test_ho_scaled = scaler_ho.transform(X_test_ho)

# Train and evaluate
svm_ho = SVC(C=1.0, kernel="rbf", gamma="scale")
svm_ho.fit(X_train_ho_scaled, y_train_ho)
test_score = svm_ho.score(X_test_ho_scaled, y_test_ho)

print(f"\nTest set size: {len(y_test_ho)} samples ({len(y_test_ho) / len(y) * 100:.1f}% of data)")
print(f"Training set size: {len(y_train_ho)} samples ({len(y_train_ho) / len(y) * 100:.1f}% of data)")
print(f"\nTest Accuracy: {test_score:.4f}")

print("\n" + "=" * 60)
print("LIMITATIONS OF HOLD-OUT")
print("=" * 60)
print("1. Performance estimate depends on the specific split")
print("2. Some data is never used for training")
print("3. Can give overly optimistic or pessimistic estimates")
print("4. Not ideal for small datasets")

## 4.2 K-Fold Cross-Validation (Manual Implementation)

**How K-Fold Cross-Validation Works:**

1. Split the dataset into K equal-sized folds
2. For each of the K iterations:
   - Use 1 fold as the test set
   - Use remaining K-1 folds as the training set
   - Train and evaluate the model
3. Average the K performance scores

**Advantages:**
- Every sample is used for both training and testing
- More robust performance estimate
- Reduces variance in the estimate

Let's implement this manually to understand the mechanics.


In [ ]:
from sklearn.model_selection import KFold

print("=" * 60)
print("K-FOLD CROSS-VALIDATION (MANUAL)")
print("=" * 60)

# Initialize K-Fold
n_splits = 5
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Store scores for each fold
fold_scores = []

# Manual iteration through folds
for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X), 1):
    # Split data
    X_train_fold, X_test_fold = X[train_idx], X[test_idx]
    y_train_fold, y_test_fold = y[train_idx], y[test_idx]

    # Scale features (IMPORTANT: fit scaler only on training data!)
    scaler_fold = StandardScaler()
    X_train_fold_scaled = scaler_fold.fit_transform(X_train_fold)
    X_test_fold_scaled = scaler_fold.transform(X_test_fold)

    # Train and evaluate
    svm_fold = SVC(C=1.0, kernel="rbf", gamma="scale")
    svm_fold.fit(X_train_fold_scaled, y_train_fold)
    score = svm_fold.score(X_test_fold_scaled, y_test_fold)

    fold_scores.append(score)
    print(f"Fold {fold_idx}: Accuracy = {score:.4f}")

# Calculate statistics
mean_score = np.mean(fold_scores)
std_score = np.std(fold_scores)

print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)
print(f"Mean Accuracy: {mean_score:.4f} (+/- {std_score:.4f})")
print(f"Min Accuracy:  {min(fold_scores):.4f}")
print(f"Max Accuracy:  {max(fold_scores):.4f}")
print(f"\nCompare with Hold-out: {test_score:.4f}")
print(f"Difference: {abs(mean_score - test_score):.4f}")

## 4.3 Using `cross_validate()`

Sklearn provides `cross_validate()` which automates cross-validation and can compute multiple metrics simultaneously. It also returns both training and test scores.


In [ ]:
from sklearn.model_selection import RepeatedKFold

rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

# Create a pipeline (combines preprocessing and model)
# This ensures scaling is done correctly within each fold
pipeline = Pipeline([("scaler", StandardScaler()), ("svm", SVC(C=1.0, kernel="rbf", gamma="scale"))])

# Define multiple scoring metrics
scoring = {"accuracy": "accuracy", "precision": "precision_weighted", "recall": "recall_weighted", "f1": "f1_weighted"}


# Perform cross-validation
cv_results = cross_validate(
    pipeline,
    X,
    y,
    cv=5,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,  # Use all available cores
)

# Convert to DataFrame for better visualization
results_df = pd.DataFrame(cv_results)

print("=" * 60)
print("CROSS_VALIDATE() RESULTS")
print("=" * 60)
print("\nDetailed Results:")
print(results_df)

print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
for metric in ["accuracy", "precision", "recall", "f1"]:
    test_scores_m = cv_results[f"test_{metric}"]
    print(f"\nTest {metric.capitalize()}:")
    print(f"  Mean: {np.mean(test_scores_m):.4f}")
    print(f"  Std:  {np.std(test_scores_m):.4f}")
    print(f"  Min:  {np.min(test_scores_m):.4f}")
    print(f"  Max:  {np.max(test_scores_m):.4f}")

## 4.4 Using `cross_val_score()`

For quick model evaluation with a single metric, `cross_val_score()` is simpler and more concise.


In [ ]:
from sklearn.model_selection import cross_val_score

print("=" * 60)
print("CROSS_VAL_SCORE() - QUICK EVALUATION")
print("=" * 60)

# Quick cross-validation with single metric
scores = cross_val_score(pipeline, X, y, cv=5, scoring="accuracy", n_jobs=-1)

print(f"\nFold Scores: {scores}")
print(f"\nMean Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")
print(
    f"95% Confidence Interval: [{scores.mean() - 1.96 * scores.std():.4f}, {scores.mean() + 1.96 * scores.std():.4f}]"
)

# Compare different models quickly
print("\n" + "=" * 60)
print("COMPARING DIFFERENT C VALUES")
print("=" * 60)

C_values_cv = [0.01, 0.1, 1, 10, 100]
for C_val in C_values_cv:
    pipeline_c = Pipeline([("scaler", StandardScaler()), ("svm", SVC(C=C_val, kernel="rbf", gamma="scale"))])
    scores_c = cross_val_score(pipeline_c, X, y, cv=5, scoring="accuracy")
    print(f"C={C_val:<6}: Mean Accuracy = {scores_c.mean():.4f} (+/- {scores_c.std():.4f})")

# 5. Hyperparameter Tuning with Cross-Validation

**Hyperparameters** are model settings that we choose before training (unlike parameters which are learned during training).

For SVM, key hyperparameters include:
- **C**: Regularization strength (inverse — larger C means less regularization)
- **kernel**: The kernel function ('linear', 'rbf', 'poly', 'sigmoid')
- **gamma**: Kernel coefficient for 'rbf', 'poly' and 'sigmoid' — controls the influence radius
- **degree**: Degree of the polynomial kernel
- **coef0**: Independent term in 'poly' and 'sigmoid' kernels

**Goal**: Find the hyperparameter combination that gives the best cross-validation performance.

**Rule of thumb**: For SVM, `C` and `gamma` should almost always be searched on a **logarithmic** grid (e.g. `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`).

## 5.1 Manual Grid Search

Grid search exhaustively tries all combinations of hyperparameter values.

**Process:**
1. Define a grid of hyperparameter values to try
2. For each combination:
   - Perform cross-validation
   - Record the mean score
3. Select the combination with the best score


In [ ]:
print("=" * 60)
print("MANUAL GRID SEARCH")
print("=" * 60)

# Define parameter grid
param_grid = {"C": [0.1, 1, 10, 100], "gamma": [0.01, 0.1, 1], "kernel": ["rbf", "linear"]}

# Store results
results_list = []

# Nested loops for grid search
total_combinations = len(param_grid["C"]) * len(param_grid["gamma"]) * len(param_grid["kernel"])

print(f"\nTesting {total_combinations} combinations...")
print("(This may take a moment...)\n")

combo_idx = 0
for C_val in param_grid["C"]:
    for gamma_val in param_grid["gamma"]:
        for kernel_val in param_grid["kernel"]:
            combo_idx += 1

            # Create pipeline with these hyperparameters
            pipe = Pipeline([("scaler", StandardScaler()), ("svm", SVC(C=C_val, gamma=gamma_val, kernel=kernel_val))])

            # Cross-validate
            scores_pipe = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
            mean_score_g = scores_pipe.mean()
            std_score_g = scores_pipe.std()

            # Store results
            results_list.append(
                {
                    "C": C_val,
                    "gamma": gamma_val,
                    "kernel": kernel_val,
                    "mean_score": mean_score_g,
                    "std_score": std_score_g,
                }
            )

            print(
                f"[{combo_idx}/{total_combinations}] C={C_val:<6} gamma={gamma_val:<6} "
                f"kernel={kernel_val:<8} => Accuracy: {mean_score_g:.4f} (+/- {std_score_g:.4f})"
            )

# Convert to DataFrame and sort
results_manual_grid = pd.DataFrame(results_list)
results_manual_grid = results_manual_grid.sort_values("mean_score", ascending=False)

print("\n" + "=" * 60)
print("TOP 5 CONFIGURATIONS")
print("=" * 60)
print(results_manual_grid.head())

# Best configuration
best_params = results_manual_grid.iloc[0]
print("\n" + "=" * 60)
print("BEST HYPERPARAMETERS")
print("=" * 60)
print(f"C:      {best_params['C']}")
print(f"gamma:  {best_params['gamma']}")
print(f"kernel: {best_params['kernel']}")
print(f"Best CV Accuracy: {best_params['mean_score']:.4f} (+/- {best_params['std_score']:.4f})")

In [ ]:
# Visualize grid search results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Heatmap for RBF kernel
data_rbf = results_manual_grid[results_manual_grid["kernel"] == "rbf"]
pivot_rbf = data_rbf.pivot(index="C", columns="gamma", values="mean_score")
sns.heatmap(pivot_rbf, annot=True, fmt=".4f", cmap="YlGnBu", ax=axes[0], cbar_kws={"label": "Accuracy"})
axes[0].set_title("Grid Search Results: RBF Kernel", fontsize=14, fontweight="bold")
axes[0].set_xlabel("gamma", fontsize=12)
axes[0].set_ylabel("C", fontsize=12)

# Plot 2: Heatmap for linear kernel
data_lin = results_manual_grid[results_manual_grid["kernel"] == "linear"]
pivot_lin = data_lin.pivot(index="C", columns="gamma", values="mean_score")
sns.heatmap(pivot_lin, annot=True, fmt=".4f", cmap="YlGnBu", ax=axes[1], cbar_kws={"label": "Accuracy"})
axes[1].set_title("Grid Search Results: Linear Kernel", fontsize=14, fontweight="bold")
axes[1].set_xlabel("gamma (ignored by linear kernel)", fontsize=12)
axes[1].set_ylabel("C", fontsize=12)

plt.tight_layout()
plt.show()

## 5.2 Manual Random Search

Random search samples random combinations from the hyperparameter space. It's more efficient than grid search when:
- The hyperparameter space is large
- Some hyperparameters don't significantly affect performance
- You have limited computational resources

**Advantage**: Can explore a wider range of values with fewer evaluations.


In [ ]:
import random

print("=" * 60)
print("MANUAL RANDOM SEARCH")
print("=" * 60)

# Define parameter distributions (wider range than grid search)
param_distributions = {
    # log-uniform over several orders of magnitude
    "C": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "gamma": [0.001, 0.01, 0.1, 1, 10, "scale", "auto"],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],  # only used for poly
}

# Number of random combinations to try
n_iterations = 20

results_random = []

print(f"\nTesting {n_iterations} random combinations...")
print("(Sampling from a larger parameter space)\n")

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)

for i in range(n_iterations):
    # Randomly sample hyperparameters
    C_val = random.choice(param_distributions["C"])
    gamma_val = random.choice(param_distributions["gamma"])
    kernel_val = random.choice(param_distributions["kernel"])
    degree_val = random.choice(param_distributions["degree"])

    # Create pipeline
    pipe = Pipeline(
        [("scaler", StandardScaler()), ("svm", SVC(C=C_val, gamma=gamma_val, kernel=kernel_val, degree=degree_val))]
    )

    # Cross-validate
    scores_rs = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    mean_score_rs = scores_rs.mean()
    std_score_rs = scores_rs.std()

    # Store results
    results_random.append(
        {
            "C": C_val,
            "gamma": gamma_val,
            "kernel": kernel_val,
            "degree": degree_val,
            "mean_score": mean_score_rs,
            "std_score": std_score_rs,
        }
    )

    print(
        f"[{i + 1:2d}/{n_iterations}] C={str(C_val):<6} gamma={str(gamma_val):<8} "
        f"kernel={kernel_val:<8} degree={degree_val} => "
        f"Accuracy: {mean_score_rs:.4f} (+/- {std_score_rs:.4f})"
    )

# Convert to DataFrame and sort
results_random_df = pd.DataFrame(results_random)
results_random_df = results_random_df.sort_values("mean_score", ascending=False)

print("\n" + "=" * 60)
print("TOP 5 RANDOM CONFIGURATIONS")
print("=" * 60)
print(results_random_df.head())

# Best configuration
best_random = results_random_df.iloc[0]
print("\n" + "=" * 60)
print("BEST HYPERPARAMETERS (RANDOM SEARCH)")
print("=" * 60)
print(f"C:      {best_random['C']}")
print(f"gamma:  {best_random['gamma']}")
print(f"kernel: {best_random['kernel']}")
print(f"degree: {best_random['degree']}")
print(f"Best CV Accuracy: {best_random['mean_score']:.4f} (+/- {best_random['std_score']:.4f})")

print("\n" + "=" * 60)
print("COMPARISON: GRID vs RANDOM SEARCH")
print("=" * 60)
print(f"Grid Search Best:   {best_params['mean_score']:.4f}")
print(f"Random Search Best: {best_random['mean_score']:.4f}")
print(f"Random search tried {n_iterations} combinations vs {total_combinations} in grid search")

## 5.3 GridSearchCV (Automated Grid Search)

Sklearn's `GridSearchCV` automates the grid search process with additional features:
- Parallel processing
- Automatic cross-validation
- Detailed results tracking
- Easy access to best parameters and scores


In [ ]:
from sklearn.model_selection import GridSearchCV

print("=" * 60)
print("GRIDSEARCHCV (AUTOMATED)")
print("=" * 60)

# Create pipeline
pipeline_gs = Pipeline([("scaler", StandardScaler()), ("svm", SVC())])

# Define parameter grid (note the prefix 'svm__' for pipeline)
param_grid_gs = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1, "scale"],
    "svm__kernel": ["rbf", "linear"],
}

# Create GridSearchCV object
grid_search = GridSearchCV(
    estimator=pipeline_gs,
    param_grid=param_grid_gs,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,  # Use all available cores
    verbose=1,
    return_train_score=True,
)

# Fit grid search
print("\nRunning GridSearchCV...\n")
grid_search.fit(X, y)

print("\n" + "=" * 60)
print("GRIDSEARCHCV RESULTS")
print("=" * 60)
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")
print(f"Best Estimator: {grid_search.best_estimator_}")

# Get detailed results
cv_results_gs = pd.DataFrame(grid_search.cv_results_)
cv_results_gs = cv_results_gs.sort_values("rank_test_score")

print("\n" + "=" * 60)
print("TOP 5 CONFIGURATIONS")
print("=" * 60)
print(cv_results_gs[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head())

In [ ]:
# Visualize GridSearchCV results
fig, ax = plt.subplots(figsize=(14, 8))

# Extract relevant data
C_list = [params["svm__C"] for params in cv_results_gs["params"]]
gamma_list = [params["svm__gamma"] for params in cv_results_gs["params"]]
kernel_list = [params["svm__kernel"] for params in cv_results_gs["params"]]
scores_gs = cv_results_gs["mean_test_score"].values

# Create a scatter plot
for kernel_val in ["rbf", "linear"]:
    for gamma_val in [0.001, 0.01, 0.1, 1, "scale"]:
        mask = [(k == kernel_val and g == gamma_val) for k, g in zip(kernel_list, gamma_list)]
        C_vals = [c for c, m in zip(C_list, mask) if m]
        scores_filtered = [s for s, m in zip(scores_gs, mask) if m]

        # Sort by C for nicer plotting
        if C_vals:
            order = np.argsort(C_vals)
            C_vals_sorted = np.array(C_vals)[order]
            scores_sorted = np.array(scores_filtered)[order]
            label = f"{kernel_val}, gamma={gamma_val}"
            ax.semilogx(C_vals_sorted, scores_sorted, "o-", label=label, linewidth=2, markersize=8)

ax.set_xlabel("C (log scale)", fontsize=12)
ax.set_ylabel("Mean CV Accuracy", fontsize=12)
ax.set_title("GridSearchCV Results: All Configurations", fontsize=14, fontweight="bold")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.4 RandomizedSearchCV (Automated Random Search)

`RandomizedSearchCV` is particularly useful when:
- The hyperparameter space is very large
- You have limited computational budget
- You want to explore continuous or very large discrete spaces

Unlike `GridSearchCV`, it samples a fixed number of candidates from parameter distributions. For SVM we use `scipy.stats.loguniform` to sample `C` and `gamma` on a log scale — this is the right thing to do given how these parameters affect the model.


In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

print("=" * 60)
print("RANDOMIZEDSEARCHCV (AUTOMATED)")
print("=" * 60)

# Create pipeline
pipeline_rs = Pipeline([("scaler", StandardScaler()), ("svm", SVC())])

# Define parameter distributions
# C and gamma are sampled log-uniformly -- the natural way to search them
param_distributions_rs = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__kernel": ["rbf", "poly", "linear"],
    "svm__degree": [2, 3, 4],
}

# Create RandomizedSearchCV object
random_search = RandomizedSearchCV(
    estimator=pipeline_rs,
    param_distributions=param_distributions_rs,
    n_iter=30,  # Number of parameter settings sampled
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
    random_state=42,
    return_train_score=True,
)

# Fit random search
print("\nRunning RandomizedSearchCV...\n")
random_search.fit(X, y)

print("\n" + "=" * 60)
print("RANDOMIZEDSEARCHCV RESULTS")
print("=" * 60)
print(f"\nBest Parameters: {random_search.best_params_}")
print(f"Best CV Score: {random_search.best_score_:.4f}")

# Get detailed results
cv_results_rs = pd.DataFrame(random_search.cv_results_)
cv_results_rs = cv_results_rs.sort_values("rank_test_score")

print("\n" + "=" * 60)
print("TOP 5 CONFIGURATIONS")
print("=" * 60)
print(cv_results_rs[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head())

print("\n" + "=" * 60)
print("COMPARISON: GRIDSEARCH vs RANDOMIZEDSEARCH")
print("=" * 60)
print(f"GridSearchCV Best Score:       {grid_search.best_score_:.4f}")
print(f"RandomizedSearchCV Best Score: {random_search.best_score_:.4f}")
print(f"\nGridSearchCV tested: {len(cv_results_gs)} combinations")
print(f"RandomizedSearchCV tested: {len(cv_results_rs)} combinations")
print("\nRandom search explored a continuous log-uniform space with fewer evaluations!")

# 6. Comprehensive Example: Complete Pipeline

Let's put everything together into a complete, production-ready pipeline that incorporates all best practices we've learned.

## Complete Workflow:
1. Load and explore data
2. Split into train/test sets
3. Build pipeline with preprocessing
4. Hyperparameter tuning with cross-validation
5. Evaluate final model on test set
6. Analyze results


In [ ]:
print("=" * 70)
print("COMPLETE SVM PIPELINE - BEST PRACTICES")
print("=" * 70)

# Step 1: Load data (using Iris for this example)
print("\n[STEP 1] Loading Data...")
X_full = iris.data
y_full = iris.target
print(f"Dataset shape: {X_full.shape}")
print(f"Classes: {np.unique(y_full)}")

# Step 2: Train/Test Split (IMPORTANT: Split BEFORE any preprocessing!)
print("\n[STEP 2] Splitting Data...")
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)
print(f"Training set: {X_train_final.shape[0]} samples")
print(f"Test set: {X_test_final.shape[0]} samples")

# Step 3: Build Pipeline
print("\n[STEP 3] Building Pipeline...")
final_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),  # Always scale for SVM!
        ("svm", SVC()),
    ]
)
print("Pipeline created: StandardScaler -> SVC")

# Step 4: Hyperparameter Tuning with Cross-Validation
print("\n[STEP 4] Hyperparameter Tuning...")
print("Using GridSearchCV with 5-fold cross-validation")

param_grid_final = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": [0.01, 0.1, 1, "scale"],
    "svm__kernel": ["rbf", "linear"],
}

grid_search_final = GridSearchCV(
    estimator=final_pipeline, param_grid=param_grid_final, cv=5, scoring="accuracy", n_jobs=-1, verbose=0
)

# Fit on TRAINING data only
grid_search_final.fit(X_train_final, y_train_final)

print(f"Best parameters found: {grid_search_final.best_params_}")
print(f"Best cross-validation score: {grid_search_final.best_score_:.4f}")

# Step 5: Evaluate on Test Set
print("\n[STEP 5] Final Model Evaluation on Test Set...")
best_model = grid_search_final.best_estimator_
y_pred_final = best_model.predict(X_test_final)

# Calculate all metrics
final_accuracy = accuracy_score(y_test_final, y_pred_final)
final_precision = precision_score(y_test_final, y_pred_final, average="weighted")
final_recall = recall_score(y_test_final, y_pred_final, average="weighted")
final_f1 = f1_score(y_test_final, y_pred_final, average="weighted")

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE")
print("=" * 70)
print(f"Test Accuracy:  {final_accuracy:.4f} ({final_accuracy * 100:.2f}%)")
print(f"Test Precision: {final_precision:.4f}")
print(f"Test Recall:    {final_recall:.4f}")
print(f"Test F1-Score:  {final_f1:.4f}")

print("\n" + "=" * 70)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(y_test_final, y_pred_final, target_names=iris.target_names))

In [ ]:
# Final confusion matrix
cm_final = confusion_matrix(y_test_final, y_pred_final)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=iris.target_names)
disp.plot(cmap="Greens", ax=ax, values_format="d")
plt.title("Final Model - Confusion Matrix", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Performance comparison table
print("\n" + "=" * 70)
print("PERFORMANCE COMPARISON: ALL APPROACHES")
print("=" * 70)

comparison_data = {
    "Approach": [
        "Hold-out (single split)",
        "Manual K-Fold CV",
        "cross_validate()",
        "GridSearchCV",
        "RandomizedSearchCV",
        "Final Tuned Model (Test)",
    ],
    "Accuracy": [
        f"{test_score:.4f}",
        f"{mean_score:.4f}",
        f"{np.mean(cv_results['test_accuracy']):.4f}",
        f"{grid_search.best_score_:.4f}",
        f"{random_search.best_score_:.4f}",
        f"{final_accuracy:.4f}",
    ],
    "Note": [
        "Single evaluation",
        "5-fold average",
        "5-fold with metrics",
        "Best of grid search",
        "Best of random search",
        "Unseen test data",
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# 7. Key Takeaways and Best Practices

## Summary of SVM

**Strengths:**
- ✅ Strong theoretical foundation (max-margin, convex optimization)
- ✅ Effective in high-dimensional spaces (even when features > samples)
- ✅ Memory-efficient at prediction time (only support vectors are stored)
- ✅ Versatile through kernels — can model almost any decision surface
- ✅ Unique global optimum — no random restarts needed
- ✅ Works well on small and medium datasets

**Weaknesses:**
- ❌ Training is slow on large datasets (roughly O(n²)-O(n³) for kernel SVMs)
- ❌ No native probability estimates — requires calibration (Platt scaling)
- ❌ Hard to interpret with non-linear kernels (black box)
- ❌ Very sensitive to C, gamma, and feature scaling
- ❌ Multi-class is handled via wrappers (OvO or OvR), not natively
- ❌ Struggles with noisy / overlapping classes unless C is tuned carefully

## Best Practices for SVM

### 1. **ALWAYS Scale Your Features**
```python
# Use StandardScaler, MinMaxScaler, or RobustScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same transformation!
```

### 2. **Use Pipelines**
```python
# Ensures proper preprocessing in cross-validation
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])
```

### 3. **Search C and gamma on a Log Scale**
- `C` and `gamma` are best explored on a log-uniform grid:
  - `C ∈ {0.001, 0.01, 0.1, 1, 10, 100, 1000}`
  - `gamma ∈ {0.0001, 0.001, 0.01, 0.1, 1, 10}`
- Linear grids will waste most of your evaluations on irrelevant values.

### 4. **Start with `SVC(kernel='rbf', C=1, gamma='scale')`**
- This is a strong baseline for most problems.
- Scikit-learn's `gamma='scale'` uses `1 / (n_features * X.var())`, which adapts to the data.

### 5. **Choose Your Kernel Carefully**
- **Linear**: high-dimensional sparse data (e.g. text, TF-IDF) — also much faster
- **RBF**: generic non-linear problems (default)
- **Polynomial**: when you expect multiplicative feature interactions of a specific degree
- **Sigmoid**: rarely the best choice — mostly of historical interest

### 6. **Mind the Cost for Large Datasets**
- For n > ~50,000 samples, kernel SVMs become impractical.
- Consider `LinearSVC` (much faster, linear-only) or switch to a different algorithm (logistic regression, gradient boosting, neural nets).
- Approximate kernel methods (`Nystroem`, `RBFSampler`) can make kernel SVMs scale.

### 7. **Handle Imbalanced Data**
- Use `class_weight='balanced'` to automatically re-weight classes inversely to their frequency.
- Combine with careful threshold tuning on the decision function.

### 8. **Cross-Validation Strategy**
- Never tune hyperparameters on the test set!
- Use nested cross-validation for unbiased performance estimates.
- Stratify splits for classification to maintain class balance.

### 9. **Performance Optimization**
- Use `n_jobs=-1` in `GridSearchCV` / `RandomizedSearchCV` for parallel processing.
- Cache the kernel matrix if you are doing many queries on the same training set (`cache_size` argument in `SVC`).

### 10. **Model Evaluation**
- Use multiple metrics (accuracy, precision, recall, F1).
- Examine the confusion matrix for misclassification patterns.
- For regression (SVR): check residual plots and use R², RMSE, and MAE together.
- Always report confidence intervals (mean ± std from CV).

## When to Use SVM

✅ **Use SVM when:**
- You have small to medium-sized datasets (up to ~50k samples)
- Features are mostly continuous and can be scaled
- Decision boundaries are expected to be non-linear
- You need a strong, well-understood baseline
- You are working in a very high-dimensional space (e.g. text classification)
- You want a model with a unique, convex global optimum

❌ **Avoid SVM when:**
- Dataset is very large (slow training)
- You need natural probability estimates
- Features are mostly categorical without a good embedding
- You need a highly interpretable model

## Common Pitfalls to Avoid

1. **Forgetting to scale features** → RBF kernel becomes dominated by large-variance features
2. **Searching C and gamma on a linear grid** → Most of the computation is wasted
3. **Using the default `C=1, gamma='scale'` blindly** → Often not optimal; at least verify with CV
4. **Leaking test data into preprocessing** → Always use a `Pipeline`
5. **Tuning hyperparameters on the test set** → Biased estimates
6. **Ignoring class imbalance** → Use `class_weight='balanced'`
7. **Using kernel SVM on hundreds of thousands of samples** → Training will never finish

## Further Reading

- [Scikit-learn SVM Documentation](https://scikit-learn.org/stable/modules/svm.html)
- [A Practical Guide to SVM Classification (Hsu, Chang, Lin)](https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf)
- [Support Vector Machine — Wikipedia](https://en.wikipedia.org/wiki/Support-vector_machine)
- [Cross-Validation Techniques](https://scikit-learn.org/stable/modules/cross_validation.html)


# 8. Homework Assignment

## Wine Quality Classification using SVM

### Objective
Apply everything you've learned about SVM to classify wine quality using the Wine Quality dataset.

### Dataset
Use the **Wine Quality dataset** from sklearn or UCI Machine Learning Repository:
- Binary classification: Good quality (score ≥ 7) vs. Not good quality (score < 7)
- Features: Various chemical properties of wine
- Available via: `sklearn.datasets` or download from UCI

### Tasks

#### Part 1: Data Exploration and Preprocessing
1. Load the wine quality dataset
2. Explore the dataset:
   - Print shape, feature names, and class distribution
   - Create visualizations (histograms, correlation matrix)
   - Check for missing values and outliers
3. Create binary classification labels (Good/Not Good quality)
4. Split data into train (70%) and test (30%) sets with stratification

#### Part 2: From-Scratch Implementation
1. Implement a linear SVM classifier from scratch using hinge loss + sub-gradient descent (similar to Section 2.1)
2. Map your labels to {-1, +1}
3. Try at least two values of `C` (e.g. 0.1 and 10) and observe how the margin changes
4. Report accuracy, precision, recall and F1-score on the held-out test set

#### Part 3: Scikit-learn Implementation with Evaluation
1. Build a pipeline with `StandardScaler` and `SVC`
2. Compare performance with:
   - Different kernels: `'linear'`, `'rbf'`, `'poly'`
   - Different `C` values on a log scale: `[0.01, 0.1, 1, 10, 100]`
   - Different `gamma` values: `['scale', 0.01, 0.1, 1]`
3. Create visualizations:
   - Accuracy vs. `C` (log-scale x-axis)
   - Confusion matrix for best model
   - If possible (2-3 features): decision boundary plot
4. Write a detailed classification report

#### Part 4: Cross-Validation
1. Implement 5-fold cross-validation manually using a for loop
2. Use `cross_validate()` with multiple scoring metrics
3. Compare CV results with hold-out results
4. Discuss which approach gives more reliable estimates and why

#### Part 5: Hyperparameter Tuning
1. Use `GridSearchCV` to find optimal hyperparameters:
   - `C`: `[0.1, 1, 10, 100]`
   - `gamma`: `[0.001, 0.01, 0.1, 1, 'scale']`
   - `kernel`: `['rbf', 'linear', 'poly']`
2. Use `RandomizedSearchCV` with:
   - `C`: `loguniform(1e-3, 1e3)`
   - `gamma`: `loguniform(1e-4, 1e1)`
   - At least 20 iterations
3. Compare GridSearch vs RandomSearch:
   - Best scores achieved
   - Number of evaluations
   - Computational time
4. Visualize the hyperparameter search results (heatmaps or log-scale plots)

#### Part 6: Final Model and Analysis
1. Train final model with best hyperparameters on full training set
2. Evaluate on test set with all metrics
3. Create a comprehensive performance report including:
   - All evaluation metrics
   - Confusion matrix
   - Comparison table of all approaches
4. **Bonus**: Discuss:
   - Which kernel worked best and why?
   - What does the ratio `#support_vectors / #training_samples` tell you?
   - What are the limitations of SVM for this problem?
   - How would you compare SVM performance with KNN from the previous notebook?
